# Python `venv` Environments — The Isolation Awakens

*Alice's enthusiasm rises as Bob's voice gets quieter.*

## Introduction — One User, Two Projects

After yesterday's visit, Alice continues explaining how the shared system interpreter works. Each time she speaks, Bob silently repeats her words in a childish imitation, then answers aloud with only “Aha,” “Yes,” or “No.” His voice gets calmer as he gets more offended; his inner imitation gets more petulant.

> **Alice:** Yesterday we saw that a user install belongs to one account, but it is still shared by every project that user runs.
>
> **Bob (thinking, in a childish imitation):** “Shared by every project! Every single one!”
>
> **Bob (aloud, quietly):** Aha.
>
> **Alice:** A `venv` gives each project its own package directory, so two projects can use incompatible dependency versions without overwriting each other.
>
> **Bob (thinking, in a childish imitation):** “Incompatible versions! Ooh, scary!”
>
> **Bob (aloud, even more calmly):** Yes.

Run this notebook in order in its [Dev Container](./README.md). It follows [The Raise of Python](../nb1_python_system_lab/python_system_lab.ipynb) and accompanies [Python `venv` Environments](../../docs/chapter-01/section-02.md). Cells run from `/workspace`; the notebook kernel remains in its own tooling environment throughout.

## Create Two Environments

> **Alice:** Look, `python3 -m venv .venv` creates a project-local environment using the base interpreter we choose!
>
> **Bob (thinking, in a childish imitation):** “Look! It makes a little environment!”
>
> **Bob (aloud, in a level voice):** Aha.
>
> **Alice:** The `python3-venv` package supplies the creation support on Ubuntu; `venv` does not install or manage Python versions.
>
> **Bob (thinking, in a childish imitation):** “It doesn't install Python versions. There, I said it too.”
>
> **Bob (aloud, calmly):** Yes.

This setup resets only the ignored `lab/` directory.

In [ ]:
%%bash
set -euxo pipefail
rm -rf lab
mkdir -p lab/legacy lab/new
/usr/bin/python3 -m venv lab/legacy/.venv
/usr/bin/python3 -m venv lab/new/.venv
ls -la lab/legacy/.venv
ls -l lab/legacy/.venv/bin/python*
cat lab/legacy/.venv/pyvenv.cfg


## Inspect the Layout

> **Alice:** The `pyvenv.cfg` file records environment settings; `sys.prefix` points to the environment, while `sys.base_prefix` points to the base Python.
>
> **Bob (thinking, in a childish imitation):** “Prefix this, prefix that. Look at me, I know two prefixes!”
>
> **Bob (aloud, softly):** Aha.
>
> **Alice:** By default, the environment excludes system and user site-packages, but still shares the base installation's standard library.
>
> **Bob (thinking, in a childish imitation):** “Still shares the standard library. What a shocking discovery.”
>
> **Bob (aloud, barely above a murmur):** Yes.

Compare the two interpreters and inspect their package locations.

In [ ]:
%%bash
set -eu
for py in /usr/bin/python3 lab/legacy/.venv/bin/python; do
  "$py" -c 'import sys, sysconfig; print("executable:", sys.executable); print("prefix:", sys.prefix); print("base prefix:", sys.base_prefix); print("stdlib:", sysconfig.get_path("stdlib")); print("packages:", sysconfig.get_path("purelib")); print()'
done
lab/legacy/.venv/bin/python -m site


## Isolate Project Dependencies

> **Alice:** Now we can install `idna` 3.7 into one environment and 3.10 into the other, using each environment's own pip.
>
> **Bob (thinking, in a childish imitation):** “One gets three point seven, the other gets three point ten. So clever!”
>
> **Bob (aloud, with measured calm):** Yes.
>
> **Alice:** See? Each project imports its own version, and neither install overwrites the other!
>
> **Bob (thinking, in a childish imitation):** “Neither overwrites the other. Gold star!”
>
> **Bob (aloud, almost serene):** Aha.

These installs need network access. No package is installed into the system interpreter.

In [ ]:
%%bash
set -euxo pipefail
lab/legacy/.venv/bin/python -m pip install 'idna==3.7'
lab/new/.venv/bin/python -m pip install 'idna==3.10'
for py in lab/legacy/.venv/bin/python lab/new/.venv/bin/python; do
  "$py" -c 'import idna; print(idna.__version__, idna.__file__)'
done
lab/legacy/.venv/bin/python -c 'import idna; assert idna.__version__ == "3.7"'
lab/new/.venv/bin/python -c 'import idna; assert idna.__version__ == "3.10"'


## Activation and Import Resolution

> **Alice:** Activation just puts the environment's `bin` directory first on that shell's `PATH`; we can also call its Python directly.
>
> **Bob (thinking, in a childish imitation):** “Just changes PATH. Or type the whole path. Either way, apparently.”
>
> **Bob (aloud, quietly and evenly):** No.
>
> **Alice:** Python then uses its own `sys.path`, and activation in one notebook shell cell does not persist in the next.
>
> **Bob (thinking, in a childish imitation):** “It doesn't persist in the next cell. I'll try not to faint.”
>
> **Bob (aloud, barely audible):** Aha.

This cell demonstrates activation and deactivation in the same shell, while checking the selected package version.

In [ ]:
%%bash
set -eu
echo "Before: $(command -v python3)"
source lab/legacy/.venv/bin/activate
echo "Activated: $(command -v python3)"
python3 -c 'import idna, sys; print(idna.__version__, idna.__file__); print(*sys.path, sep="\n")'
deactivate
echo "After: $(command -v python3)"


## Epilogue — Disposable, Not Portable

> **Alice:** We have two dependency versions, no conflict, and the same base interpreter underneath. Isn't that neat?
>
> **Bob (thinking, in a childish imitation):** “Isn't that neat? Isn't it just wonderful?”
>
> **Bob (aloud, almost whispering):** Yes.
>
> **Alice:** Recreate environments instead of copying them: scripts and configuration can contain absolute paths, and the base Python must still exist.
>
> **Bob (thinking, in a childish imitation):** “Recreate them. Don't copy them. I get it.”
>
> **Bob (aloud, perfectly composed):** Aha.
>
> **Alice:** Record project dependencies in a project definition or requirements file; `venv` itself does not lock them.
>
> **Bob (thinking, in a childish imitation):** “Write down the dependencies too. Anything else, Professor Alice?”
>
> **Bob (aloud, very calmly):** No.

Bob's voice is calm, but he is more offended than when they started. The [technical companion](../../docs/chapter-01/section-02.md) covers the directory layout, platform differences, and the PyGuard project workflow.